In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# ===== EEGNet定義 =====
class EEGNet(nn.Module):
    def __init__(self, n_classes=5, n_channels=4, n_times=34,
                 F1=8, D=2, F2=16, dropout=0.5):
        super().__init__()

        # Block 1: Temporal Conv
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(F1),
            # Depthwise Conv（チャンネル方向）
            nn.Conv2d(F1, F1*D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1*D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )

        # Block 2: Separable Conv
        self.block2 = nn.Sequential(
            nn.Conv2d(F1*D, F1*D, (1, 16), padding=(0, 8), groups=F1*D, bias=False),
            nn.Conv2d(F1*D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )

        self.classifier = nn.Linear(F2 * n_times, n_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# ===== データセット =====
class EEGDataset(Dataset):
    def __init__(self, spec_dir, labels_df):
        self.samples = []
        self.labels = []

        for _, row in labels_df.iterrows():
            sub = int(row['subject'])
            ses = int(row['song_id'])
            fname = f"sub{sub:03d}_ses{ses:02d}.npy"
            fpath = os.path.join(spec_dir, fname)

            if not os.path.exists(fpath):
                continue

            data = np.load(fpath).astype(np.float32)
            # (4ch, freq, time) → 正規化
            data = (data - data.mean()) / (data.std() + 1e-8)
            self.samples.append(data)
            self.labels.append(int(row['enjoyment']) - 1)  # 0-indexed

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x = torch.tensor(self.samples[idx]).unsqueeze(0)  # (1, 4, freq, time)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

# ===== 学習 =====
def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item()
            correct += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

# ===== メイン =====
data_root = r"C:\Users\hiro2\Documents\EEG_project\data"
spec_dir  = os.path.join(data_root, "spectrograms")
labels_df = pd.read_csv(os.path.join(data_root, "features_clean.csv"))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")

# データセット作成
dataset = EEGDataset(spec_dir, labels_df)
print(f"データ数: {len(dataset)}")

# サンプルのshape確認
x0, y0 = dataset[0]
print(f"入力shape: {x0.shape}, ラベル: {y0}")

# 時間bins数を自動取得
n_times = x0.shape[-1] // 8  # AvgPool後のサイズ
print(f"n_times（FC入力）: {n_times}")

# モデル作成
model = EEGNet(
    n_classes=5,
    n_channels=4,
    n_times=n_times,
    dropout=0.5
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"総パラメータ数: {total_params:,}")

# Leave-One-Subject-Out CV
logo = LeaveOneGroupOut()
groups = labels_df['subject'].values[:len(dataset)]
all_labels = [dataset[i][1].item() for i in range(len(dataset))]
all_accs = []

for fold, (train_idx, test_idx) in enumerate(
        logo.split(range(len(dataset)), all_labels, groups)):

    train_sub = DataLoader(
        torch.utils.data.Subset(dataset, train_idx),
        batch_size=16, shuffle=True)
    test_sub = DataLoader(
        torch.utils.data.Subset(dataset, test_idx),
        batch_size=16)

    model = EEGNet(n_classes=5, n_channels=4,
                   n_times=n_times, dropout=0.5).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    # 50エポック学習
    for epoch in range(50):
        train(model, train_sub, optimizer, criterion, device)

    _, acc = evaluate(model, test_sub, criterion, device)
    all_accs.append(acc)
    print(f"Fold {fold+1:2d} (sub-{fold+1:03d}): acc={acc:.3f}")

print(f"\n平均accuracy: {np.mean(all_accs):.3f} ± {np.std(all_accs):.3f}")
print(f"チャンス精度: {1/5:.3f} (5クラス)")